# Public data: Alvheim-area ownership and production (Sodir FactPages)

No account needed. Downloads are cached in `data/public` (git-ignored). Use this to check
licence shares for `licence_shares.csv`, and to find past tie-in start dates for the
interference study.

In [ ]:
import plotly.express as px

from alvheim_tiein import sodir
from alvheim_tiein.config import Settings

public = Settings.from_env().data_dir / "public"
public.mkdir(parents=True, exist_ok=True)


def table(name: str):
    path = public / f"{name}.csv"
    if not path.exists():
        path.write_text(sodir.fetch_text(name), encoding="utf-8")
    return sodir.parse_table(path.read_text(encoding="utf-8"))


licensees = table("field_licensee_hst")
production = table("field_production_monthly")
sodir.current_field_licensees(licensees)

In [ ]:
prod = sodir.field_monthly(production)
px.area(
    prod, x="period", y="prfPrdOeNetMillSm3", color="prfInformationCarrier",
    title="Saleable production per field (mill Sm3 oe per month)",
).show()

First production month per field: the start-ups of earlier tie-ins are natural experiments
for how a new field affects the wells already flowing.

In [ ]:
prod[prod["prfPrdOeNetMillSm3"] > 0].groupby("prfInformationCarrier")["period"].min().sort_values()

In [ ]:
area = licensees[licensees["fldName"].isin(sodir.ALVHEIM_AREA_FIELDS)]
m = prod.merge(area, left_on="prfInformationCarrier", right_on="fldName")
m = m[(m["fldLicenseeFrom"] <= m["period"]) & (m["fldLicenseeTo"].isna() | (m["fldLicenseeTo"] >= m["period"]))]
m["company_oe"] = m["prfPrdOeNetMillSm3"] * m["fldCompanyShare"] / 100.0
by_company = m.groupby(["period", "cmpLongName"], as_index=False)["company_oe"].sum()
px.area(
    by_company, x="period", y="company_oe", color="cmpLongName",
    title="Equity share of saleable production by licensee (mill Sm3 oe per month)",
).show()